A hospital supervisor is to create a schedule for 4 nurses over a 5-day period. Each day is divided into 3 eight-hour shifts, so that there are 15 shifts. Each nurse can request for specific shifts, that is, $c_{ij} = 1$ if Nurse $i$ ($i=1:4$) requests for Shift $j$ ($j=1:15$), and $c_{ij} = 0$ otherwise. How to create a schedule to maximise the number of requests that are met, subject to: 
1. Each shift is assigned to exactly one nurse.
2. Every day, no nurse works more than one shift.
3. Each nurse is assigned to at least three shifts, and no more than four shifts, during the 5-day period.
4. Each nurse cannot take consecutive shifts during the 5-day period.  

Define $x_{ij} = 1$ if Nurse $i$ is scheduled to Shift $j$, and $x_{ij} = 0$ otherwise. The problem can be formulated as:
\begin{align*}
\max_{x_{ij}}\,\,&\sum_{i=1}^4\sum_{j=1}^{15} c_{ij}x_{ij},\\
\text{s.t.  }&\sum_{i=1}^4x_{ij}=1,\quad j=1,\cdots, 15,\\
&\sum_{j=3(k-1)+1}^{3k}x_{ij}\le 1, \quad i=1,\cdots, 4, \,\,k=1,\cdots, 5,\\
&3\le\sum_{j=1}^{15}x_{ij}\le 4, \quad i=1,\cdots, 4,\\
&x_{ij}+x_{i,j+1}\le 1,\quad i=1,\cdots, 4, \,\, j=1,\cdots,14,\\
&x_{ij}\in\{0,1\},\quad i=1,\cdots, 4, \,\, j=1,\cdots,15.
\end{align*}

In [1]:
from ortools.linear_solver import pywraplp

In [3]:
# parameters
n_nurses = 4
n_days = 5
n_shifts_per_day = 3
n_shifts = n_days * n_shifts_per_day  # 15 shifts

# nurse requests: 1 = requested, 0 = not requested
c = [
    [1,	1,	1,	0,	0,	0,	0,	1,	0,	0,	0,	0,	0,	0,	1],  # Nurse 0
    [0,	1,	1,	1,	1,	0,	0,	0,	0,	0,	0,	0,	0,	0,	1],  # Nurse 1
    [0,	0,	0,	0,	1,	0,	0,	1,	0,	0,	1,	1,	0,	0,	1],  # Nurse 2
    [0,	1,	1,	0,	0,	0,	0,	0,	0,	0,	0,	1,	1,	0,	1]   # Nurse 3
]

In [7]:
# Create Model
solver = pywraplp.Solver.CreateSolver('SCIP')

In [8]:
# Decision variables 
x = {}
for i in range(n_nurses):
    for j in range(n_shifts):
        x[i,j] = solver.IntVar(0, 1, f'x_{i}_{j}')

In [9]:
# Objective: Maximise requests satisfied
objective = solver.Objective()
for i in range(n_nurses):
    for j in range(n_shifts):
        objective.SetCoefficient(x[i,j], c[i][j])
objective.SetMaximization()

In [10]:
# Constraints

# 1. Each shift is assigned to exactly one nurse
for j in range(n_shifts):
    solver.Add(solver.Sum(x[i,j] for i in range(n_nurses)) == 1)

In [11]:
# 2. No nurse works more than one shift per day
for i in range(n_nurses):
    for d in range(n_days):
        day_shifts = [d*n_shifts_per_day + s for s in range(n_shifts_per_day)]
        solver.Add(solver.Sum(x[i,j] for j in day_shifts) <= 1)

In [12]:
# 3. Each nurse works at least 3 and at most 4 shifts total
for i in range(n_nurses):
    solver.Add(solver.Sum(x[i,j] for j in range(n_shifts)) >= 3)
    solver.Add(solver.Sum(x[i,j] for j in range(n_shifts)) <= 4)

In [13]:
# 4. No consecutive shifts for the same nurse
for i in range(n_nurses):
    for j in range(n_shifts - 1):
        solver.Add(x[i,j] + x[i,j+1] <= 1)

In [15]:
# Solve
status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
    print("Optimal schedule found!\n")
    print("Optimal objective value =", solver.Objective().Value())
    schedule = [[0]*n_shifts for _ in range(n_nurses)]
    for i in range(n_nurses):
        for j in range(n_shifts):
            schedule[i][j] = int(x[i,j].solution_value())
    
    # Print schedule 
    for i in range(n_nurses):
        print(f"Nurse {i}: ", schedule[i])
else:
    print("No optimal solution found.")

Optimal schedule found!

Optimal objective value = 9.0
Nurse 0:  [1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0]
Nurse 1:  [0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0]
Nurse 2:  [0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1]
Nurse 3:  [0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0]
